# Phase 2, Task 3 — Split, Train, Evaluate

You have a clean `X` (691406 × 78) and `y`. Time to train a model and score it — but there's a new wrinkle vs Phase 1.

**Phase 1 gave you two separate files** (KDDTrain+ and KDDTest+), so the train/test split was done *for* you. **Here you have ONE file (Wednesday).** So *you* must carve out a test set yourself — hold back a slice of rows the model never trains on, to grade it honestly.

**New tool — `train_test_split`:** randomly splits your data into a training part and a testing part (e.g. 80% / 20%). The model learns on the 80% and is graded on the unseen 20%.

**New term — `stratify`:** remember the imbalance (63% benign)? A random split *could* accidentally put too few attacks in the test set. `stratify=y` forces the split to keep the **same benign/attack ratio** in both train and test — so the grade is fair.

## Step 0 — Rebuild clean X and y
Fresh notebook → rebuild the clean data from Task 2.

**Your job:** reload the Wednesday CSV, strip column names, Inf→NaN→dropna, build `is_attack`, and make `X` and `y`.

**Hints:**
- This is exactly your Task 2 code — copy it. Just make sure the outputs are named `X` and `y` (not `X_train`).
- Quick check: `X.shape` should be `(691406, 78)`.

In [ ]:
# TODO: rebuild clean X and y from the Wednesday file (reuse your Task 2 code)


## Step 1 — Split into train and test yourself
**Your job:** split `X`, `y` into `X_train, X_test, y_train, y_test` — 80% train / 20% test, stratified.

**Hints:**
- `from sklearn.model_selection import train_test_split`
- `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`
- `test_size=0.2` = hold back 20% for testing. `random_state=42` = reproducible split. `stratify=y` = keep the benign/attack ratio balanced.
- Sanity check: print `X_train.shape` and `X_test.shape` (should be ~553k and ~138k rows), and compare `y_train.value_counts(normalize=True)` vs `y_test.value_counts(normalize=True)` — the ratios should match closely (that's stratify working).

In [ ]:
# TODO: train_test_split into train/test (stratified); print the shapes and check the ratios


## Step 2 — Train a RandomForest
Same model as Phase 1.

**Your job:** create a `RandomForestClassifier` and `.fit()` it on `X_train, y_train`.

**Hints:**
- `from sklearn.ensemble import RandomForestClassifier`
- `model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)`
- `model.fit(X_train, y_train)` — with ~553k rows this may take 30-90 seconds. That's normal.

In [ ]:
# TODO: create and fit a RandomForestClassifier on the training data


## Step 3 — Predict and read the scorecard
**Your job:** predict on `X_test` and print a `classification_report`.

**Hints:**
- `y_pred = model.predict(X_test)`
- `from sklearn.metrics import classification_report`
- `print(classification_report(y_test, y_pred))`
- The attack class is `1`. Look at its precision AND recall.
- (Reminder in `reference/METRICS_GLOSSARY.md` if any term is fuzzy.)

In [ ]:
# TODO: predict on X_test and print the classification_report


## Step 4 — Interpret (read this carefully — it's the twist)
You'll probably see **very high precision AND recall** on the attack class (often 0.99+). After Phase 1's painful 0.61 recall, that feels like a miracle. It isn't — and understanding *why* is the whole point.

**Why so high here?** Your test set was carved from the **same Wednesday file** as your training set. So every attack *type* in the test (DoS Hulk, GoldenEye, slowloris, Slowhttptest) also appeared in training — the model has **seen examples of each**. Recall is high because there are no surprises. This is a **within-distribution** test.

**Phase 1's recall was low** because its test file (KDDTest+) deliberately contained attack *types absent* from training — a **cross-distribution** test. That's the hard, realistic case.

**So the honest question (jot your answer in `LEARNING_LOG.md`):** is this high score proof our model would catch a *brand-new* attack it has never seen? Why or why not? What would be a tougher, more realistic way to test it? (Hint: we have 7 other days of data with *different* attack types...)

That tougher test — train on one day's attacks, test on a *different* day's attacks — is the next task, and it's where Phase 2 earns its keep.

### ✅ You pass Phase 2 Task 3 when:
1. You split the single file into train/test yourself, stratified, and can explain why `stratify` matters with imbalanced data.
2. A RandomForest is trained on the training slice and scored on the held-out test slice.
3. You can read the attack class's precision and recall from the report.
4. You can explain *why* these scores are so much higher than Phase 1 — the within- vs cross-distribution difference.

Paste me your `classification_report` and your answer to the Step 4 question, and I'll review — then Task 4 is the real test: a different day's attacks.